In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy rustworkx scipy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Quantum Approximate Optimization Algorithm

*Nutzungsschätzung: 22 Minute uf emm Heron r3 Prozessor (HINWEIS: Des isch nur e Schätzung. Dini Laufzeit ka variiere.)*
## Hintergrund {#Hintergrund}
Des Tutorial zeigt d'Implementierung vom **Quantum Approximate Optimization Algorithm (QAOA)** – ere hybriden (quanten-klassische) iterative Methode – im Kontext vo Qiskit-Patterns. Mir löse zunächst des **Maximum-Cut** (oder **Max-Cut**) Problem für e kleini Graphe un lerne dann, wie mer's uf Utility-Skala ausführt. Alli Hardware-Ausführunge im Tutorial sollte innerhalb vom Zeitlimit für dr frei zugängliche Open Plan funktioniere.

Des Max-Cut-Problem isch e Optimierungsproblem, wo schwer z'löse isch (genauer gsagt isch es e *NP-hartes* Problem) un e Reihe verschiedeni Anwendunge im Clustering, Netzwerkwissenschaft un statistischer Physik hän. Des Tutorial betrachtet e Graphe vo Knoten, wo durch Kante verbunde sin, un zielt druf ab, d'Knoten in zwei Menge z'partitioniere, so dass d'Anzahl vo de durch dene Schnitt durchquerte Kante maximiert wird.

![Illustration of a max-cut problem](../docs/images/tutorials/quantum-approximate-optimization-algorithm/maxcut-illustration.avif)
## Voraussetzunge {#Voraussetzunge}

Schau vor em Aafange vo diesem Tutorial, dass mir des Folgende installiert hän:
- Qiskit SDK v1.0 oder neuer, mit [Visualisierungs](https://docs.quantum.ibm.com/api/qiskit/visualization)-Unterstützung
- Qiskit Runtime v0.22 oder neuer (`pip install qiskit-ibm-runtime`)

Zusätzlich brauche mir Zugang z'ere Instanz uf dr [IBM Quantum Platform](/guides/cloud-setup). Beachte, dass des Tutorial nit im Open Plan ausgeführt werre ka, weil's Workloads mit [Sessions](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/session) ausführt, wo nur mit Premium Plan-Zugang verfügbar sin.
## Einrichtung {#Einrichtung}

In [1]:
import matplotlib.pyplot as plt
import rustworkx as rx
from rustworkx.visualization import mpl_draw as draw_graph
import numpy as np
from scipy.optimize import minimize
from collections import defaultdict
from typing import Sequence


from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import QAOAAnsatz
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import Session, EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler

## Teil I. QAOA im kleini Maßstab {#Teil-I}
Dr erschte Teil vo diesem Tutorial verwendet e kleins Max-Cut-Problem, um d'Schritte z'r Lösung vo emm Optimierungsproblem mit emm Quantecomputer z'veranschauliche.

Um e bessere Kontext z'geh, bevor mir des Problem uf e Quantealgorithmus abbilde, kenne mir besser verstehe, wie des Max-Cut-Problem z'emm klassische kombinatorische Optimierungsproblem wird, indem mir zunächst d'Minimierung vo ere Funktion $f(x)$ betrachte

$$
\min_{x\in {0, 1}^n}f(x),
$$

wobei d'Eingab $x$ e Vektor isch, wo seine Komponente jedem Knoten vo emm Graphe entspreche. Dann wird jede vo dene Komponente uf entweder $0$ oder $1$ beschränkt (wo repräsentiere, ob sie im Schnitt enthalte sin oder nit). Dene kleinskalige Beispielfall verwendet e Graphe mit $n=5$ Knoten.

Mir kenne e Funktion vo emm Knotepaar $i,j$ schreibe, wo azeigt, ob d'entsprechend Kant $(i,j)$ im Schnitt leit. Zum Beispiel isch d'Funktion $x_i + x_j - 2 x_i x_j$ genau dann 1, wenn entweder $x_i$ oder $x_j$ gleich 1 isch (was bedeutet, dass d'Kant im Schnitt leit) un sonst null. Des Problem, d'Kante im Schnitt z'maximiere, ka formuliert werre als

$$
\max_{x\in {0, 1}^n} \sum_{(i,j)} x_i + x_j - 2 x_i x_j,
$$

was als Minimierung umgschriebe werre ka in dr Form

$$
\min_{x\in {0, 1}^n} \sum_{(i,j)}  2 x_i x_j - x_i - x_j.
$$

Des Minimum vo $f(x)$ in diesem Fall leit vor, wenn d'Anzahl vo de durch dr Schnitt durchquerte Kante maximal isch. Wie mir sehe kenne, hät des no nix mit Quantecomputing z'tun. Mir müsse des Problem in öppis umformuliere, wo e Quantecomputer verstehe ka.
Initialisiere mr des Problem, indem mir e Graphe mit $n=5$ Knoten erstelle.

In [2]:
n_small = 5

graph = rx.PyGraph()
graph.add_nodes_from(np.arange(0, n_small, 1))
edge_list = [
    (0, 1, 1.0),
    (0, 2, 1.0),
    (0, 4, 1.0),
    (1, 2, 1.0),
    (2, 3, 1.0),
    (3, 4, 1.0),
]
graph.add_edges_from(edge_list)
draw_graph(graph, node_size=600, with_labels=True)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/6ced6bea-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/6ced6bea-0.avif)

### Schritt 1: Klassische Eingabe uf e Quanteproblem abbilde {#Schritt-1}

Dr erschte Schritt vom Pattern besteht drin, des klassische Problem (Graphe) uf quantemechanischi **Schaltkreise** un **Operatoren** abbilde. Dafür sin drei Hauptschritte z'unternehme:

1. Verwendung vo ere Reihe mathematischer Umformulierunge, um des Problem mithilfe vo dr Notation vo Quadratic Unconstrained Binary Optimization (QUBO) Problemen darzustelle.
2. Umformulierung vom Optimierungsproblem als Hamilton-Operator, für wo dr Grundzustand dr Lösung entspricht, wo d'Kostenfunktion minimiert.
3. Erstellung vo emm Quanteschaltkreis, wo dr Grundzustand vo diesem Hamilton-Operator über e Prozess ähnlich emm Quantum Annealing vorbereitet.

**Hinweis:** In dr QAOA-Methodik wolle mir letztendlich e Operator (**Hamilton-Operator**) hän, wo d'**Kostenfunktion** vo unserm hybriden Algorithmus darstellt, sowie e parametrisierten Schaltkreis (**Ansatz**), wo Quantezustände mit Kandidatenlösunge für des Problem darstellt. Mir kenne aus dene Kandidatezuständ samplen un sie dann mit dr Kostenfunktion bewerte.

#### Graph &rarr; Optimierungsproblem {#Graph-Optimierungsproblem}

Dr erschte Schritt vo dr Abbildung isch e Notationsänderung. Des Folgende drückt des Problem in QUBO-Notation aus:

$$
\min_{x\in {0, 1}^n}x^T Q x,
$$

wobei $Q$ e $n\times n$ Matrix vo reelle Zahle isch, $n$ dr Anzahl vo de Knoten im Graphe entspricht, $x$ dr obe eingeführte Vektor vo binäre Variablen isch un $x^T$ d'Transponierte vom Vektor $x$ bezeichnet.

```
Maximize
 -2*x_0*x_1 - 2*x_0*x_2 - 2*x_0*x_4 - 2*x_1*x_2 - 2*x_2*x_3 - 2*x_3*x_4 + 3*x_0
 + 2*x_1 + 3*x_2 + 2*x_3 + 2*x_4

Subject to
  No constraints

  Binary variables (5)
    x_0 x_1 x_2 x_3 x_4
```
### Optimierungsproblem &rarr; Hamilton-Operator {#Optimierungsproblem-Hamilton-Operator}

Mir kenne dann des QUBO-Problem als **Hamilton-Operator** umformuliere (do e Matrix, wo d'Energie vo emm System darstellt):

$$
H_C=\sum_{ij}Q_{ij}Z_iZ_j + \sum_i b_iZ_i.
$$

<details>
<summary>
**Umformulierungsschritte vom QAOA-Problem zum Hamilton-Operator**
</summary>

Um z'zeige, wie des QAOA-Problem uf diese Weis umgschriebe werre ka, ersetze mir zunächst d'binäre Variablen $x_i$ durch e neue Satz vo Variablen $z_i\in{-1, 1}$ über

$$
x_i = \frac{1-z_i}{2}.
$$

Do kenne mir sehe, dass wenn $x_i$ gleich $0$ isch, dann $z_i$ gleich $1$ si muss. Wenn d'$x_i$ durch d'$z_i$ im Optimierungsproblem ($x^TQx$) ersetzt werre, ka e äquivalenti Formulierung erhalte werre.

$$
x^TQx=\sum_{ij}Q_{ij}x_ix_j \\ =\frac{1}{4}\sum_{ij}Q_{ij}(1-z_i)(1-z_j) \\=\frac{1}{4}\sum_{ij}Q_{ij}z_iz_j-\frac{1}{4}\sum_{ij}(Q_{ij}+Q_{ji})z_i + \frac{n^2}{4}.
$$

Wenn mir jetzt $b_i=-\sum_{j}(Q_{ij}+Q_{ji})$ definiere, dr Vorfaktor entferne un dr konstante Term $n^2$ weglasse, erhalte mir d'beide äquivalente Formulierunge vo demselbe Optimierungsproblem.

$$
\min_{x\in{0,1}^n} x^TQx\Longleftrightarrow \min_{z\in{-1,1}^n}z^TQz + b^Tz
$$

Do hängt $b$ vo $Q$ ab. Beachte, dass mir z'r Erlangung vo $z^TQz + b^Tz$ dr Faktor 1/4 un e konstante Offset vo $n^2$ wegglosse hän, wo bi dr Optimierung kei Rolle spiele.

Um jetzt e Quanteformulierung vom Problem z'erhalte, erhebe mir d'Variablen $z_i$ z'ere Pauli $Z$ Matrix, wie ere $2\times 2$ Matrix vo dr Form

$$
Z_i = \begin{pmatrix}1 & 0 \\ 0 & -1\end{pmatrix}.
$$

Wenn mir dene Matrize im obige Optimierungsproblem iisetze, erhalte mir dr folgende Hamilton-Operator

$$
H_C=\sum_{ij}Q_{ij}Z_iZ_j + \sum_i b_iZ_i.
$$

*Beachte au, dass d'$Z$ Matrize in dr Rechebereich vom Quantecomputer iibettet sin, des heißt in e Hilbert-Raum vo dr Größe $2^n\times 2^n$. Deshalbe sollte mir Terme wie $Z_iZ_j$ als des Tensorprodukt $Z_i\otimes Z_j$ verstehe, wo in dr $2^n\times 2^n$ Hilbert-Raum iibettet isch. Zum Beispiel wird in emm Problem mit fünf Entscheidungsvariablen dr Term $Z_1Z_3$ als $I\otimes Z_3\otimes I\otimes Z_1\otimes I$ verstande, wobei $I$ d'$2\times 2$ Einheitsmatrix isch.*
</details>

Dene Hamilton-Operator wird als **Kostenfunktions-Hamilton-Operator** bezeichnet. Er hät d'Eigenschaft, dass si Grundzustand dr Lösung entspricht, wo d'**Kostenfunktion $f(x)$ minimiert**.
Um des Optimierungsproblem z'löse, müsse mir jetzt dr Grundzustand vo $H_C$ (oder e Zustand mit hoher Überlappung damit) uf emm Quantecomputer präpariere. Des Sampeln aus diesem Zustand wird dann mit hoher Wahrscheinlichkeit d'Lösung z'$min~f(x)$ liefere.
Betrachte mir jetzt dr Hamilton-Operator $H_C$ für des **Max-Cut** Problem. Jedem Knoten vom Graphe wird e Qubit im Zustand $|0\rangle$ oder $|1\rangle$ zugeordnet, wobei dr Wert d'Menge agibt, z'der dr Knoten ghört. Des Ziel vom Problem isch es, d'Anzahl vo de Kante $(v_1, v_2)$ z'maximiere, für wo $v_1 = |0\rangle$ un $v_2 = |1\rangle$ gilt, oder umgekehrt. Wenn mir dr $Z$ Operator jedem Qubit zuordne, wobei

$$
    Z|0\rangle = |0\rangle \qquad Z|1\rangle = -|1\rangle
$$

dann ghört e Kant $(v_1, v_2)$ zum Schnitt, wenn dr Eigenwert vo $(Z_1|v_1\rangle) \cdot (Z_2|v_2\rangle) = -1$ isch; mit andere Worten, d'mit $v_1$ un $v_2$ assoziierten Qubits sin unterschiedlich. Ebenso ghört $(v_1, v_2)$ nit zum Schnitt, wenn dr Eigenwert vo $(Z_1|v_1\rangle) \cdot (Z_2|v_2\rangle) = 1$ isch. Beachte, dass uns dr genaue Qubit-Zustand, wo jedem Knoten zugeordnet isch, nit interessiert, sondern nur, ob sie über e Kant nüber gleich sin oder nit. Des Max-Cut-Problem verlangt vo uns, e Zuordnung vo de Qubits uf d'Knoten z'finde, so dass dr Eigenwert vom folgende Hamilton-Operator minimiert wird
$$
    H_C = \sum_{(i,j) \in e} Q_{ij} \cdot Z_i Z_j.
$$

Mit andere Worten, $b_i = 0$ für alli $i$ im Max-Cut-Problem. Dr Wert vo $Q_{ij}$ bezeichnet des Gewicht vo dr Kant. In diesem Tutorial betrachte mir e ungewichtete Graphe, des heißt $Q_{ij} = 1.0$ für alli $i, j$.

In [3]:
def build_max_cut_paulis(
    graph: rx.PyGraph,
) -> list[tuple[str, list[int], float]]:
    """Convert graph edges to a list of ZZ Pauli terms.

    The returned list is in the sparse format expected by
    ``SparsePauliOp.from_sparse_list``: each element is
    ``(pauli_string, qubit_indices, coefficient)``.
    """
    pauli_list = []
    for edge in list(graph.edge_list()):
        weight = graph.get_edge_data(edge[0], edge[1])
        pauli_list.append(("ZZ", [edge[0], edge[1]], weight))
    return pauli_list


max_cut_paulis = build_max_cut_paulis(graph)
cost_hamiltonian = SparsePauliOp.from_sparse_list(max_cut_paulis, n_small)
print("Cost Function Hamiltonian:", cost_hamiltonian)

Cost Function Hamiltonian: SparsePauliOp(['IIIZZ', 'IIZIZ', 'ZIIIZ', 'IIZZI', 'IZZII', 'ZZIII'],
              coeffs=[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j])


#### Build the QAOA ansatz circuit

Use `QAOAAnsatz` to construct the parametrized QAOA circuit from the cost Hamiltonian. Here we use `reps=2` (two QAOA layers, four parameters: $\beta_0, \beta_1, \gamma_0, \gamma_1$).

In [4]:
circuit = QAOAAnsatz(cost_operator=cost_hamiltonian, reps=2)
circuit.measure_all()

circuit.draw("mpl")

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/7bd8c6d4-f40f-4a11-a440-0b26d9021b53-0.avif" alt="Output of the previous code cell" />

In [5]:
circuit.parameters

ParameterView([ParameterVectorElement(β[0]), ParameterVectorElement(β[1]), ParameterVectorElement(γ[0]), ParameterVectorElement(γ[1])])

### Step 2: Optimize problem for quantum hardware execution

Transpile the abstract circuit into hardware-native instructions. This step handles qubit mapping, gate decomposition, routing, and error suppression. See the transpilation [documentation](/docs/guides/transpile) for more information.

In [6]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
print(backend)

# Create pass manager for transpilation. Level 3 is the most aggressive
# preset: slower to transpile, but produces shorter circuits that are
# more robust to hardware noise.
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

candidate_circuit = pm.run(circuit)
candidate_circuit.draw("mpl", fold=False, idle_wires=False)

<IBMBackend('ibm_pittsburgh')>


<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/3f28a422-805c-4d3d-b5f6-62539e9133bd-1.avif" alt="Output of the previous code cell" />

### Step 3: Execute using Qiskit primitives

The QAOA optimization loop runs inside a Qiskit Runtime [session](/docs/guides/execution-modes) to keep the device reserved across iterations. An Estimator evaluates $\langle H_C \rangle$ at each step, and a classical optimizer (COBYLA) updates the parameters until convergence.

![Illustration showing the behavior of Single job, Batch, and Session runtime modes.](../docs/images/tutorials/quantum-approximate-optimization-algorithm/runtime-modes.avif)

Define initial parameters and run the optimization loop:

In [7]:
# QAOA doesn't prescribe principled default angles — any bounded choice
# works as a warm start for problems this small. beta and gamma are
# periodic (beta in [0, pi] and gamma in [0, 2*pi] modulo the underlying
# Pauli-rotation periods), and pi/2 and pi are just midpoints of those
# ranges. For harder problems you would typically warm start from known
# good angles or transfer parameters from smaller instances.
initial_gamma = np.pi
initial_beta = np.pi / 2
init_params = [initial_beta, initial_beta, initial_gamma, initial_gamma]

In [8]:
def cost_func_estimator(params, ansatz, hamiltonian, estimator):
    # transform the observable defined on virtual qubits to
    # an observable defined on all physical qubits
    isa_hamiltonian = hamiltonian.apply_layout(ansatz.layout)

    pub = (ansatz, isa_hamiltonian, params)
    job = estimator.run([pub])

    results = job.result()[0]
    cost = results.data.evs

    objective_func_vals.append(cost)

    return cost

In [9]:
objective_func_vals = []  # Global variable
with Session(backend=backend) as session:
    # If using qiskit-ibm-runtime<0.24.0, change `mode=` to `session=`
    estimator = Estimator(mode=session)
    estimator.options.default_shots = 1000

    # Set simple error suppression/mitigation options
    estimator.options.dynamical_decoupling.enable = True
    estimator.options.dynamical_decoupling.sequence_type = "XY4"
    estimator.options.twirling.enable_gates = True
    estimator.options.twirling.num_randomizations = "auto"
    estimator.options.environment.job_tags = ["TUT_QAOA"]

    result = minimize(
        cost_func_estimator,
        init_params,
        args=(candidate_circuit, cost_hamiltonian, estimator),
        method="COBYLA",
        tol=1e-2,
    )
    print(result)

 message: Return from COBYLA because the trust region radius reaches its lower bound.
 success: True
  status: 0
     fun: -2.0402211719947774
       x: [ 3.041e+00  1.212e+00  2.081e+00  4.471e+00]
    nfev: 36
   maxcv: 0.0


![Output of the previous code cell](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/3f28a422-805c-4d3d-b5f6-62539e9133bd-1.avif)

### Schritt 3: Ausführung mit Qiskit Primitives {#Schritt-3}
Im QAOA-Workflow werre d'optimale QAOA-Parameter in ere iterativen Optimierungsschleife gfunde, wo e Reihe vo Schaltkreisbewertunge ausführt un e klassische Optimierer verwendet, um d'optimale $\beta_k$ un $\gamma_k$ Parameter z'finde. Dene Ausführungsschleife wird über d'folgende Schritte ausgeführt:

1. Definiere vo de initialen Parameter
2. Instanziierung vo ere neue `Session`, wo d'Optimierungsschleife un des Primitive enthält, wo zum Samplen vom Schaltkreis verwendet wird
3. Sobald e optimale Parametersatz gfunde isch, führe mir dr Schaltkreis e letschts Mol aus, um e finale Verteilung z'erhalte, wo im Post-Processing-Schritt verwendet wird.
#### Schaltkreis mit initialen Parametern definiere {#Schaltkreis-init-Parameter}
Mir beginne mit willkürlich gwählte Parametern.

In [10]:
plt.figure(figsize=(12, 6))
plt.plot(objective_func_vals)
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.show()

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/e14ecc92-0.avif" alt="Output of the previous code cell" />

#### Backend un Ausführungs-Primitive definiere {#Backend-Primitive}
Mir verwende d'**Qiskit Runtime Primitives**, um mit IBM&reg; Backends z'interagiere. D'beide Primitives sin Sampler un Estimator, un d'Wahl vom Primitive hängt davo ab, welchi Art vo Messung mir uf emm Quantecomputer ausführe möchte. Für d'Minimierung vo $H_C$ verwende mir dr Estimator, weil d'Messung vo dr Kostenfunktion eifach dr Erwartungswert vo $\langle H_C \rangle$ isch.
#### Ausführe {#Ausfuehre}

D'Primitives biete e Vielzahl vo [Ausführungsmodi](/guides/execution-modes) z'r Planung vo Workloads uf Quantegeräte, un e QAOA-Workflow lauft iterativ in ere Session.

![Illustration showing the behavior of Single job, Batch, and Session runtime modes.](../docs/images/tutorials/quantum-approximate-optimization-algorithm/runtime-modes.avif)

Mir kenne d'sampler-basierte Kostenfunktion in d'SciPy-Minimierungsroutine iistecke, um d'optimale Parameter z'finde.

In [11]:
optimized_circuit = candidate_circuit.assign_parameters(result.x)
optimized_circuit.draw("mpl", fold=False, idle_wires=False)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/2989e76e-4296-4dd8-b065-2b8fced064cf-0.avif" alt="Output of the previous code cell" />

In [12]:
# If using qiskit-ibm-runtime<0.24.0, change `mode=` to `backend=`
sampler = Sampler(mode=backend)
sampler.options.default_shots = 10000

# Set simple error suppression/mitigation options
sampler.options.dynamical_decoupling.enable = True
sampler.options.dynamical_decoupling.sequence_type = "XY4"
sampler.options.twirling.enable_gates = True
sampler.options.twirling.num_randomizations = "auto"

sampler.options.environment.job_tags = ["TUT_QAOA"]

pub = (optimized_circuit,)
job = sampler.run([pub], shots=int(1e4))
counts_int = job.result()[0].data.meas.get_int_counts()
counts_bin = job.result()[0].data.meas.get_counts()
shots = sum(counts_int.values())
final_distribution_int = {key: val / shots for key, val in counts_int.items()}
final_distribution_bin = {key: val / shots for key, val in counts_bin.items()}
print(final_distribution_int)

{18: 0.039, 5: 0.0665, 20: 0.0973, 29: 0.0063, 9: 0.0899, 13: 0.0379, 2: 0.0047, 1: 0.0153, 11: 0.0932, 14: 0.0327, 12: 0.0314, 25: 0.0193, 21: 0.0398, 6: 0.0224, 4: 0.0197, 10: 0.0387, 3: 0.0181, 26: 0.07, 17: 0.0327, 19: 0.0332, 22: 0.0914, 24: 0.007, 0: 0.0033, 8: 0.0066, 30: 0.0158, 28: 0.0169, 27: 0.0222, 16: 0.0073, 7: 0.0057, 23: 0.0062, 15: 0.0054, 31: 0.0041}


### Step 4: Post-process and return result in desired classical format

Extract the most likely bitstring from the sampled distribution. This represents the best cut found by QAOA.

In [13]:
# auxiliary functions to sample most likely bitstring
def to_bitstring(integer, num_bits):
    result = np.binary_repr(integer, width=num_bits)
    return [int(digit) for digit in result]


keys = list(final_distribution_int.keys())
values = list(final_distribution_int.values())
most_likely = keys[np.argmax(np.abs(values))]
most_likely_bitstring = to_bitstring(most_likely, len(graph))
most_likely_bitstring.reverse()

print("Result bitstring:", most_likely_bitstring)

Result bitstring: [0, 0, 1, 0, 1]


In [14]:
plt.rcParams.update({"font.size": 10})
final_bits = final_distribution_bin
values = np.abs(list(final_bits.values()))
top_4_values = sorted(values, reverse=True)[:4]
positions = []
for value in top_4_values:
    positions.append(np.where(values == value)[0])
fig = plt.figure(figsize=(11, 6))
ax = fig.add_subplot(1, 1, 1)
plt.xticks(rotation=45)
plt.title("Result Distribution")
plt.xlabel("Bitstrings (reversed)")
plt.ylabel("Probability")
ax.bar(list(final_bits.keys()), list(final_bits.values()), color="tab:grey")
for p in positions:
    ax.get_children()[int(p[0])].set_color("tab:purple")
plt.show()

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/650875e9-adbc-43bd-9505-556be2566278-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/e14ecc92-0.avif)

Sobald mir d'optimale Parameter für dr Schaltkreis gfunde hän, kenne mir dene Parameter zuweise un d'mit de optimierte Parametern erhalteni finale Verteilung sampeln. Do soll des *Sampler* Primitive verwendet werre, weil es d'Wahrscheinlichkeitsverteilung vo Bitstring-Messunge isch, wo emm optimale Schnitt vom Graphe entspreche.

**Hinweis:** Des bedeutet, e Quantezustand $\psi$ im Computer z'präpariere un ihn dann z'messe. E Messung wird dr Zustand in e einzelne Berechnungsbasiszustand kollabiere loh - zum Beispiel `010101110000...` - wo ere Kandidatenlösung $x$ für unser ursprünglichs Optimierungsproblem ($\max f(x)$ oder $\min f(x)$ je nach Aufgab) entspricht.

In [15]:
# auxiliary function to plot graphs
def plot_result(G, x):
    colors = ["tab:grey" if i == 0 else "tab:purple" for i in x]
    pos, _default_axes = rx.spring_layout(G), plt.axes(frameon=True)
    rx.visualization.mpl_draw(
        G, node_color=colors, node_size=100, alpha=0.8, pos=pos
    )


plot_result(graph, most_likely_bitstring)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/33135970-8bc4-4fb2-ab87-08726a432ce4-0.avif" alt="Output of the previous code cell" />

Now, calculate the value of the cut:

In [16]:
def evaluate_sample(x: Sequence[int], graph: rx.PyGraph) -> float:
    assert len(x) == len(
        list(graph.nodes())
    ), "The length of x must coincide with the number of nodes in the graph."
    return sum(
        x[u] * (1 - x[v]) + x[v] * (1 - x[u])
        for u, v in list(graph.edge_list())
    )


cut_value = evaluate_sample(most_likely_bitstring, graph)
print("The value of the cut is:", cut_value)

The value of the cut is: 5


For a graph this small, the true optimum is easy to brute-force, so you can double-check the results by comparing the QAOA result against the exact answer.

In [17]:
# Classical baseline: enumerate all 2**n_small bitstrings and take the best cut.
def brute_force_max_cut(graph: rx.PyGraph) -> tuple[int, list[int]]:
    n = len(list(graph.nodes()))
    best_cut = -1
    best_x: list[int] = []
    for i in range(2**n):
        x = [(i >> k) & 1 for k in range(n)]
        cut = evaluate_sample(x, graph)
        if cut > best_cut:
            best_cut = int(cut)
            best_x = x
    return best_cut, best_x


classical_best, classical_x = brute_force_max_cut(graph)
print(f"Classical optimum (brute force): {classical_best}")
print(f"QAOA cut value:                  {cut_value}")

Classical optimum (brute force): 5
QAOA cut value:                  5


### Schritt 4: Nachbearbeitung un Rückgab vom Ergebnis im gwünschte klassische Format {#Schritt-4}

Dr Nachbearbeitungsschritt interpretiert d'Sampling-Ausgab, um e Lösung für des ursprüngliche Problem z'rückz'geh. In diesem Fall sin mir an emm Bitstring mit dr höchste Wahrscheinlichkeit interessiert, weil dene dr optimale Schnitt bestimmt. D'Symmetrie im Problem erlaube vier mögliche Lösunge, un dr Sampling-Prozess wird eine davo mit ere öppis höhere Wahrscheinlichkeit z'rückgeh, aber mir kenne in dr unte dargestellte Verteilung sehe, dass vier vo de Bitstrings deutlich wahrscheinlicher sin als dr Rest.

In [ ]:
# Precomputed parity lookup table: _PARITY[b] = +1 if popcount(b) is even, else -1.
# We use this to vectorize expectation-value evaluation across all Pauli terms.
_PARITY = np.array(
    [-1 if bin(i).count("1") % 2 else 1 for i in range(256)],
    dtype=np.complex128,
)


def evaluate_sparse_pauli(state: int, observable: SparsePauliOp) -> complex:
    """Expectation value of a SparsePauliOp on a single computational-basis state.

    For a Z-only observable (which QAOA cost Hamiltonians are, after the
    QUBO-to-Hamiltonian mapping), the eigenvalue of each Pauli term on a
    computational-basis state is simply (-1)**popcount(z_mask AND state),
    i.e., the parity of the bitwise-AND of the term's Z-support and the
    measured bitstring.

    This routine packs the Z-support of every Pauli term into bytes, ANDs
    them against the measured state in a single vectorized op, and looks up
    the parity in _PARITY. For a 100-qubit / ~hundreds-of-terms Hamiltonian
    over 10_000 samples, this is dramatically faster than calling
    SparsePauliOp.expectation_value per sample.
    """
    packed_uint8 = np.packbits(observable.paulis.z, axis=1, bitorder="little")
    state_bytes = np.frombuffer(
        state.to_bytes(packed_uint8.shape[1], "little"), dtype=np.uint8
    )
    reduced = np.bitwise_xor.reduce(packed_uint8 & state_bytes, axis=1)
    return np.sum(observable.coeffs * _PARITY[reduced])


def best_solution(samples, hamiltonian):
    """Return the sampled bitstring (as int) with the lowest Hamiltonian cost."""
    min_cost = float("inf")
    min_sol = None
    for bit_str in samples.keys():
        candidate_sol = int(bit_str)
        fval = evaluate_sparse_pauli(candidate_sol, hamiltonian).real
        if fval <= min_cost:
            min_cost = fval
            min_sol = candidate_sol
    return min_sol


def _plot_cdf(objective_values: dict, ax, color):
    x_vals = sorted(objective_values.keys(), reverse=True)
    y_vals = np.cumsum([objective_values[x] for x in x_vals])
    ax.plot(x_vals, y_vals, color=color)


def plot_cdf(dist, ax, title):
    _plot_cdf(dist, ax, "C1")
    ax.vlines(min(list(dist.keys())), 0, 1, "C1", linestyle="--")
    ax.set_title(title)
    ax.set_xlabel("Objective function value")
    ax.set_ylabel("Cumulative distribution function")
    ax.grid(alpha=0.3)


def samples_to_objective_values(samples, hamiltonian):
    """Convert the samples to values of the objective function."""
    objective_values = defaultdict(float)
    for bit_str, prob in samples.items():
        candidate_sol = int(bit_str)
        fval = evaluate_sparse_pauli(candidate_sol, hamiltonian).real
        objective_values[fval] += prob
    return objective_values

**Step 1**: Build the graph, cost Hamiltonian, and ansatz.

In [19]:
# Step 1: build the 100-node graph, cost Hamiltonian, and QAOA ansatz.
n_large = 100
graph_100 = rx.PyGraph()
graph_100.add_nodes_from(np.arange(0, n_large, 1))
elist = []
for edge in backend.coupling_map:
    if edge[0] < n_large and edge[1] < n_large:
        elist.append((edge[0], edge[1], 1.0))
graph_100.add_edges_from(elist)

max_cut_paulis_100 = build_max_cut_paulis(graph_100)
cost_hamiltonian_100 = SparsePauliOp.from_sparse_list(
    max_cut_paulis_100, n_large
)

circuit_100 = QAOAAnsatz(cost_operator=cost_hamiltonian_100, reps=1)
circuit_100.measure_all()

**Step 2**: Transpile for the selected hardware backend.

In [20]:
# Step 2: transpile for hardware.
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)
candidate_circuit_100 = pm.run(circuit_100)

![Output of the previous code cell](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/650875e9-adbc-43bd-9505-556be2566278-0.avif)

#### Beste Schnitt visualisiere {#Beste-Schnitt-visualisiere}

Aus emm optimale Bitstring kenne mir dann dene Schnitt uf emm ursprüngliche Graphe visualisiere.

In [21]:
# Step 3: run the QAOA optimization loop on the device, then sample the
# final distribution with the optimized parameters.
initial_gamma = np.pi
initial_beta = np.pi / 2
init_params = [initial_beta, initial_gamma]

objective_func_vals = []  # Global variable
with Session(backend=backend) as session:
    estimator = Estimator(mode=session)
    estimator.options.default_shots = 1000

    # Set simple error suppression/mitigation options
    estimator.options.dynamical_decoupling.enable = True
    estimator.options.dynamical_decoupling.sequence_type = "XY4"
    estimator.options.twirling.enable_gates = True
    estimator.options.twirling.num_randomizations = "auto"
    estimator.options.environment.job_tags = ["TUT_QAOA"]

    result = minimize(
        cost_func_estimator,
        init_params,
        args=(candidate_circuit_100, cost_hamiltonian_100, estimator),
        method="COBYLA",
    )
    print(result)

# Assign optimal parameters and sample the final distribution.
optimized_circuit_100 = candidate_circuit_100.assign_parameters(result.x)

sampler = Sampler(mode=backend)
sampler.options.default_shots = 10000

# Set simple error suppression/mitigation options
sampler.options.dynamical_decoupling.enable = True
sampler.options.dynamical_decoupling.sequence_type = "XY4"
sampler.options.twirling.enable_gates = True
sampler.options.twirling.num_randomizations = "auto"

# Add a unique tag to the job execution
sampler.options.environment.job_tags = ["TUT_QAOA"]

pub = (optimized_circuit_100,)
job = sampler.run([pub], shots=int(1e4))

counts_int = job.result()[0].data.meas.get_int_counts()
shots = sum(counts_int.values())
final_distribution_100_int = {
    key: val / shots for key, val in counts_int.items()
}

 message: Return from COBYLA because the trust region radius reaches its lower bound.
 success: True
  status: 0
     fun: -17.172689238986344
       x: [ 2.574e+00  4.166e+00]
    nfev: 28
   maxcv: 0.0


![Output of the previous code cell](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/33135970-8bc4-4fb2-ab87-08726a432ce4-0.avif)

Un berechne mir dr Wert vom Schnitt:

In [22]:
# Step 4: find the best-cost sample and evaluate its cut value.
best_sol_100 = best_solution(final_distribution_100_int, cost_hamiltonian_100)
best_sol_bitstring_100 = to_bitstring(int(best_sol_100), len(graph_100))
best_sol_bitstring_100.reverse()

print("Result bitstring:", best_sol_bitstring_100)

cut_value_100 = evaluate_sample(best_sol_bitstring_100, graph_100)
print("The value of the cut is:", cut_value_100)

Result bitstring: [1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0]
The value of the cut is: 156


Check that the cost minimized in the optimization loop has converged, and visualize results.

In [23]:
# Plot convergence
plt.figure(figsize=(12, 6))
plt.plot(objective_func_vals)
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.show()

# Visualize the cut
plot_result(graph_100, best_sol_bitstring_100)

# Plot cumulative distribution function
result_dist = samples_to_objective_values(
    final_distribution_100_int, cost_hamiltonian_100
)
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
plot_cdf(result_dist, ax, backend.name)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/large-scale-viz-0.avif" alt="Output of the previous code cell" />

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/large-scale-viz-1.avif" alt="Output of the previous code cell" />

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/large-scale-viz-2.avif" alt="Output of the previous code cell" />

## Teil II. Skaliere mir's hoch! {#Teil-II}

Mir hän Zugang z'viele Geräte mit über 100 Qubits uf dr IBM Quantum&reg; Platform. Wähle mir eis aus, uf wem Max-Cut uf emm 100-Knoten-gewichtete Graphe glöst werre soll. Des isch e Problem im "Utility-Maßstab". D'Schritte zum Aufbau vom Workflow sin d'gleiche wie obe, jedoch mit emm viel größere Graphe.